# Lab 04 — Build a Tiny Decoder-Only Transformer
**Goal:** Implement a minimal causal self-attention block and train it on a tiny corpus.

Free-first: run on CPU or free Colab where possible.

Method: **predict → run → inspect → break → decide**.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)
T,C=8,32
x=torch.randn(2,T,C)
print("input shape:", x.shape)


## Predict before running
Before running, predict:
- What shape should Q, K, and V have?
- Why must future-token probabilities be zero?
- Why does every attention row sum to one?


In [ ]:
class TinyAttention(nn.Module):
    def __init__(self,C,heads=4):
        super().__init__()
        self.qkv=nn.Linear(C,3*C,bias=False)
        self.proj=nn.Linear(C,C,bias=False)
        self.h=heads
        self.register_buffer("mask",torch.tril(torch.ones(T,T)).bool())
    def forward(self,x):
        B,Tc,Cc=x.shape
        q,k,v=self.qkv(x).chunk(3,dim=-1)
        q=q.view(B,Tc,self.h,Cc//self.h).transpose(1,2)
        k=k.view(B,Tc,self.h,Cc//self.h).transpose(1,2)
        v=v.view(B,Tc,self.h,Cc//self.h).transpose(1,2)
        a=(q@k.transpose(-2,-1))/(Cc//self.h)**0.5
        a=a.masked_fill(~self.mask[:Tc,:Tc],float("-inf"))
        a=F.softmax(a,dim=-1)
        y=(a@v).transpose(1,2).contiguous().view(B,Tc,Cc)
        return self.proj(y)
m=TinyAttention(C)
y=m(x)
print("output:",y.shape)


## Explain the result
The causal mask preserves autoregressive training: position t can use positions ≤ t, never the future target. Softmax converts scores into a distribution, while the head dimension determines the matrix-multiplication cost.


In [ ]:
# Inspect one attention matrix row. Each row should sum to ~1 and must not put mass on future tokens.
with torch.no_grad():
    q,k,v=m.qkv(x).chunk(3,dim=-1)
    q=q.view(2,T,m.h,C//m.h).transpose(1,2)
    k=k.view(2,T,m.h,C//m.h).transpose(1,2)
    a=(q@k.transpose(-2,-1))/(C//m.h)**0.5
    a=a.masked_fill(~m.mask[:T,:T],float("-inf"))
    a=F.softmax(a,dim=-1)
print("row sums:", a[0,0].sum(-1))
print("future attention in last row:", a[0,0,-1].tolist())


## Critical-thinking answers
The important engineering skill is shape accounting. A Transformer is many matrix operations composed with residual paths; errors often appear first as dimension mismatches or incorrect masking. Learn to reason with (batch, sequence, hidden, heads, head_dim) before relying on framework abstractions.

### Sources
https://arxiv.org/abs/1706.03762


## Deepening: parameter count is not enough

This section upgrades the notebook from a runnable demo to a measurable experiment. Predict before executing.


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
configs = [
    {"d_model":128, "layers":2, "heads":4, "vocab":2000},
    {"d_model":256, "layers":4, "heads":8, "vocab":4000},
]

for c in configs:
    d, V, n = c["d_model"], c["vocab"], c["layers"]
    approx_block = 4*d*d + 2*d*(4*d)
    approx = V*d + n*approx_block + d*V
    print(c, "approx params:", f"{approx:,}")


## Break experiment

Keep parameter count approximately similar while changing depth, width, and sequence length.

Ask which resource changes:

weights, activations, attention memory, or all three.

### Required artifact

Annotate every tensor shape for B=2, L=128, d_model=256, heads=8.

Use templates/EXPERIMENT_CARD.md to record the run.


## 6. Research exercise — real text, real context windows, same Transformer

Sample documents from `HuggingFaceFW/fineweb` and measure how many examples survive when packed into context windows of 128, 512, and 2048 tokens. Then use the observed lengths to reason about padding waste and causal-attention work.


In [ ]:
!pip -q install -U transformers datasets
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np

tok=AutoTokenizer.from_pretrained("sshleifer/tiny-gpt2")
ds=load_dataset("HuggingFaceFW/fineweb","sample-10BT",split="train",streaming=True)
lengths=[]
for row in ds:
    t=row.get("text","")
    if t:
        lengths.append(len(tok.encode(t,add_special_tokens=False)))
    if len(lengths)>=300: break

print("documents:",len(lengths))
for L in [128,512,2048]:
    kept=sum(1 for n in lengths if n>=L)
    padded=sum(max(0,L-n) for n in lengths if n<L)
    print(f"context={L:4d} documents reaching full context={kept:3d} total padding if one-doc-per-window={padded:,}")
print("p50/p95 lengths:",np.percentile(lengths,[50,95]).astype(int))


<details><summary><strong>Expected findings / answer</strong></summary>

The same model architecture can see very different effective workloads depending on context length. Short documents create padding waste if each document receives a fixed window; aggressive packing can improve utilization but changes boundaries and masking. Longer windows also increase attention work. The practical design is therefore a combination of **packing policy + context length + causal masking**, not simply “choose the largest context.”
</details>
